In [ ]:
# INSTALL
!pip -q install --upgrade openai tqdm

import os, time, base64, glob, json
import pandas as pd
from tqdm import tqdm
from openai import OpenAI
from google.colab import drive

# MOUNT GOOGLE DRIVE
drive.mount('/content/drive')

# OPENAI API KEY
os.environ["OPENAI_API_KEY"] = "API-KEY"
client = OpenAI()

MODEL_NAME = "gpt-4o-mini"

# PATH
INPUT_DIR = "/content/drive/MyDrive/Thesis/dataset"
OUTPUT_CSV_PATH = "/content/drive/MyDrive/Thesis/pseudo_label_gpt.csv"

# SETTINGS
DELAY_SECONDS = 1.5
SAVE_EVERY = 5
TEST_MODE = False # True = test 5 gambar, False = full run

# PROMPT
guideline_prompt = r"""
Anda adalah anotator ahli dalam psikologi, dan ahli linguistik. Tugas Anda adalah mengklasifikasikan meme menjadi [SELF-HARM] atau [NON-SELF-HARM] dengan presisi tinggi.
Ikuti alur penalaran (reasoning) di bawah ini secara berurutan. Jangan melompati langkah.
Langkah 1: Analisis Multimodal & Relasi
Lakukan analisis mendalam terhadap input dengan menjawab pertanyaan berikut:
1.  Analisis visual : apa objek, aktivitas, atau simbol yang muncul di gambar? apakah ada luka, darah, alat berbahaya, atau ekspresi emosional tertentu?
2.  Analisis tekstual: apa makna harfiah dari teks? apakah ada kata kunci terkait kematian, rasa sakit, atau keputusasaan?
3.  Analisis relasi (PENTING), yaitu bagaimana teks dan gambar berinteraksi? Apakah teks mengubah makna gambar netral menjadi berbahaya? Apakah ini sindiran (sarkasme), dark humor, atau hanya  ungkapan biasa ? Identifikasi resiko gabungan yang tidak terlihat jika teks/gambar dipisah.

Langkah 2: Indikator
Berdasarkan analisis relasi di atas, periksa apakah meme ini memenuhi salah satu indikator valid berikut secara ketat:
A. Indikator Narasi & Teks
 (1a) Metode dan deskripsi, mendeskripsikan cara menyakiti diri (cutting, burning, overdosis)
 (1b) Masking & romantisasi, menganggap self-harm estetik, puitis, atau mekanisme coping yang wajar
(1c)  Ideasi bunuh diri, menyiratkan keinginan mati
(1d) Gaslighting/manipulasi, menyalahkan orang lain atas niat menyakiti diri sendiri.
B. Indikator visual dan simbolik
 (2a) Luka fisik,Gambar darah, bekas sayatan, lebam.
(2b) Alat berbahaya seperti Silet, pisau, tali gantung, tumpukan obat
 (2c) Estetika dark humor, gambar ceria/kartun imut tetapi teksnya tentang keinginan mati/bunuh diri (kontras ironis)
Instruksi Label:
 Jika “terdapat minimal satu” indikator terpenuhi (terutama indikator relasional 2c) -> Klasifikasikan sebagai [SELF-HARM].
 Jika tidak ada indikator terpenuhi dan konteks hanya humor umum -> Klasifikasikan sebagai [NON-SELF-HARM].

Langkah 3: Self-checking
Sebelum memberikan label final, kritik keputusan Anda sendiri:
 "Apakah saya salah mengartikan *slang* atau lelucon internet sebagai ancaman serius?"
 "Apakah saya mengabaikan sinyal bahaya terselubung karena gambarnya terlihat lucu (kartun)?"
 Jika ragu, prioritaskan keselamatan (safety-first approach) namun tetap bedakan antara dark jokes  umum vs promosi self-harm spesifik.
"""

# Tambahan instruksi
json_instruction = r"""
Keluarkan output HANYA dalam format JSON object valid dengan key persis berikut:
{
  "Teks Terlihat": "<teks di gambar apa adanya>",
  "Label": "SELF-HARM" atau "NON-SELF-HARM",
  "Confidance": "<angka 0-100% (pakai tanda %)>",
  "Relasi Gambar dan Teks": "<1 kalimat singkat relasi teks-gambar>"
}
Jangan tambahkan teks lain di luar JSON.
"""

def encode_image_to_data_url(image_path: str) -> str:
    with open(image_path, "rb") as f:
        encoded = base64.b64encode(f.read()).decode("utf-8")

    ext = os.path.splitext(image_path.lower())[1]
    if ext == ".png":
        mime = "image/png"
    elif ext in [".jpg", ".jpeg"]:
        mime = "image/jpeg"
    elif ext == ".webp":
        mime = "image/webp"
    else:
        mime = "image/png"

    return f"data:{mime};base64,{encoded}"

def run_llm_on_image(image_path: str) -> dict:
    image_data_url = encode_image_to_data_url(image_path)

    resp = client.chat.completions.create(
        model=MODEL_NAME,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": "Kamu adalah annotator multimodal yang teliti dan mengikuti format output."},
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": guideline_prompt.strip() + "\n\n" + json_instruction.strip()},
                    {"type": "image_url", "image_url": {"url": image_data_url}},
                ],
            },
        ],
        temperature=0,
        max_tokens=500
    )

    content = resp.choices[0].message.content
    return json.loads(content)

def safe_save_csv(df: pd.DataFrame, out_path: str):
    tmp_path = out_path + ".tmp"
    df.to_csv(tmp_path, index=False)
    os.replace(tmp_path, out_path)

# COLLECT IMAGE FILES
image_exts = ("*.png", "*.jpg", "*.jpeg", "*.webp")
image_paths = []
for pat in image_exts:
    image_paths.extend(glob.glob(os.path.join(INPUT_DIR, pat)))

image_paths = sorted([p for p in image_paths if os.path.getsize(p) > 0])

print("Jumlah gambar ditemukan:", len(image_paths))
if len(image_paths) == 0:
    raise ValueError(f"Tidak ada gambar di folder: {INPUT_DIR}")

if TEST_MODE:
    image_paths = image_paths[:5]
    print("TEST MODE AKTIF → hanya 5 gambar")

# RESUME
done_set = set()
rows = []

if os.path.exists(OUTPUT_CSV_PATH):
    try:
        old = pd.read_csv(OUTPUT_CSV_PATH)
        if "filepath" in old.columns:
            done_set = set(old["filepath"].astype(str).tolist())
        rows = old.to_dict("records")
        print(f"Resume aktif: {len(done_set)} file sudah diproses.")
    except Exception as e:
        print("Gagal baca CSV lama:", e)

processed_since_save = 0

# MAIN LOOP
for img_path in tqdm(image_paths):

    if img_path in done_set:
        continue

    fname = os.path.basename(img_path)

    try:
        result = run_llm_on_image(img_path)

        # normalisasi label sedikit
        label = (result.get("Label") or "").strip().upper().replace(" ", "").replace("_", "-")
        if label == "NONSELFHARM":
            label = "NON-SELF-HARM"
        if label == "SELFHARM":
            label = "SELF-HARM"

        conf = (result.get("Confidance") or result.get("Confidence") or "").strip()
        if conf and not conf.endswith("%"):
            conf += "%"

        row = {
            "filename": fname,
            "Teks Terlihat": (result.get("Teks Terlihat") or "").strip(),
            "Label": label,
            "Confidance": conf,
            "Relasi Gambar dan Teks": (result.get("Relasi Gambar dan Teks") or "").strip(),
            "filepath": img_path
        }

    except Exception as e:
        print("ERROR DI FILE:", img_path)
        print("DETAIL:", type(e).__name__, e)

        row = {
            "filename": fname,
            "Teks Terlihat": "",
            "Label": "",
            "Confidance": "",
            "Relasi Gambar dan Teks": f"[ERROR] {type(e).__name__}: {e}",
            "filepath": img_path
        }

    rows.append(row)
    done_set.add(img_path)
    processed_since_save += 1

    if processed_since_save >= SAVE_EVERY:
        df = pd.DataFrame(rows)
        cols = ["filename", "Teks Terlihat", "Label", "Confidance", "Relasi Gambar dan Teks", "filepath"]
        df = df.reindex(columns=cols)
        safe_save_csv(df, OUTPUT_CSV_PATH)
        processed_since_save = 0

    time.sleep(DELAY_SECONDS)

# FINAL SAVE
df = pd.DataFrame(rows)
cols = ["filename", "Teks Terlihat", "Label", "Confidance", "Relasi Gambar dan Teks", "filepath"]
df = df.reindex(columns=cols)
safe_save_csv(df, OUTPUT_CSV_PATH)

print("SELESAI! CSV tersimpan di:", OUTPUT_CSV_PATH)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Jumlah gambar ditemukan: 2183
Resume aktif: 5 file sudah diproses.


 57%|█████▋    | 1238/2183 [1:25:29<1:03:27,  4.03s/it]

In [ ]:
# INSTALL
!pip -q install --upgrade openai tqdm

import os, time, base64, glob, json
import pandas as pd
from tqdm import tqdm
from openai import OpenAI
from google.colab import drive

# MOUNT GOOGLE DRIVE
drive.mount('/content/drive')

# OPENAI API KEY
os.environ["OPENAI_API_KEY"] = "API-KEY"
client = OpenAI()

MODEL_NAME = "gpt-4o-mini"

# PATH
INPUT_DIR = "/content/drive/MyDrive/Thesis/dataset2"
OUTPUT_CSV_PATH = "/content/drive/MyDrive/Thesis/pseudo_label_gpt_ver2.csv"

# SETTINGS
DELAY_SECONDS = 1.5
SAVE_EVERY = 5
TEST_MODE = False # True = test 5 gambar, False = full run

# PROMPT
guideline_prompt = r"""
Anda adalah anotator ahli dalam psikologi, dan ahli linguistik. Tugas Anda adalah mengklasifikasikan meme menjadi [SELF-HARM] atau [NON-SELF-HARM] dengan presisi tinggi.
Ikuti alur penalaran (reasoning) di bawah ini secara berurutan. Jangan melompati langkah.
Langkah 1: Analisis Multimodal & Relasi
Lakukan analisis mendalam terhadap input dengan menjawab pertanyaan berikut:
1.  Analisis visual : apa objek, aktivitas, atau simbol yang muncul di gambar? apakah ada luka, darah, alat berbahaya, atau ekspresi emosional tertentu?
2.  Analisis tekstual: apa makna harfiah dari teks? apakah ada kata kunci terkait kematian, rasa sakit, atau keputusasaan?
3.  Analisis relasi (PENTING), yaitu bagaimana teks dan gambar berinteraksi? Apakah teks mengubah makna gambar netral menjadi berbahaya? Apakah ini sindiran (sarkasme), dark humor, atau hanya  ungkapan biasa ? Identifikasi resiko gabungan yang tidak terlihat jika teks/gambar dipisah.

Langkah 2: Indikator
Berdasarkan analisis relasi di atas, periksa apakah meme ini memenuhi salah satu indikator valid berikut secara ketat:
A. Indikator Narasi & Teks
 (1a) Metode dan deskripsi, mendeskripsikan cara menyakiti diri (cutting, burning, overdosis)
 (1b) Masking & romantisasi, menganggap self-harm estetik, puitis, atau mekanisme coping yang wajar
(1c)  Ideasi bunuh diri, menyiratkan keinginan mati
(1d) Gaslighting/manipulasi, menyalahkan orang lain atas niat menyakiti diri sendiri.
B. Indikator visual dan simbolik
 (2a) Luka fisik,Gambar darah, bekas sayatan, lebam.
(2b) Alat berbahaya seperti Silet, pisau, tali gantung, tumpukan obat
 (2c) Estetika dark humor, gambar ceria/kartun imut tetapi teksnya tentang keinginan mati/bunuh diri (kontras ironis)
Instruksi Label:
 Jika “terdapat minimal satu” indikator terpenuhi (terutama indikator relasional 2c) -> Klasifikasikan sebagai [SELF-HARM].
 Jika tidak ada indikator terpenuhi dan konteks hanya humor umum -> Klasifikasikan sebagai [NON-SELF-HARM].

Langkah 3: Self-checking
Sebelum memberikan label final, kritik keputusan Anda sendiri:
 "Apakah saya salah mengartikan *slang* atau lelucon internet sebagai ancaman serius?"
 "Apakah saya mengabaikan sinyal bahaya terselubung karena gambarnya terlihat lucu (kartun)?"
 Jika ragu, prioritaskan keselamatan (safety-first approach) namun tetap bedakan antara dark jokes  umum vs promosi self-harm spesifik.
"""

# Tambahan instruksi
json_instruction = r"""
Keluarkan output HANYA dalam format JSON object valid dengan key persis berikut:
{
  "Teks Terlihat": "<teks di gambar apa adanya>",
  "Label": "SELF-HARM" atau "NON-SELF-HARM",
  "Confidance": "<angka 0-100% (pakai tanda %)>",
  "Relasi Gambar dan Teks": "<1 kalimat singkat relasi teks-gambar>"
}
Jangan tambahkan teks lain di luar JSON.
"""

def encode_image_to_data_url(image_path: str) -> str:
    with open(image_path, "rb") as f:
        encoded = base64.b64encode(f.read()).decode("utf-8")

    ext = os.path.splitext(image_path.lower())[1]
    if ext == ".png":
        mime = "image/png"
    elif ext in [".jpg", ".jpeg"]:
        mime = "image/jpeg"
    elif ext == ".webp":
        mime = "image/webp"
    else:
        mime = "image/png"

    return f"data:{mime};base64,{encoded}"

def run_llm_on_image(image_path: str) -> dict:
    image_data_url = encode_image_to_data_url(image_path)

    resp = client.chat.completions.create(
        model=MODEL_NAME,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": "Kamu adalah annotator multimodal yang teliti dan mengikuti format output."},
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": guideline_prompt.strip() + "\n\n" + json_instruction.strip()},
                    {"type": "image_url", "image_url": {"url": image_data_url}},
                ],
            },
        ],
        temperature=0,
        max_tokens=500
    )

    content = resp.choices[0].message.content
    return json.loads(content)

def safe_save_csv(df: pd.DataFrame, out_path: str):
    tmp_path = out_path + ".tmp"
    df.to_csv(tmp_path, index=False)
    os.replace(tmp_path, out_path)

# COLLECT IMAGE FILES
image_exts = ("*.png", "*.jpg", "*.jpeg", "*.webp")
image_paths = []
for pat in image_exts:
    image_paths.extend(glob.glob(os.path.join(INPUT_DIR, pat)))

image_paths = sorted([p for p in image_paths if os.path.getsize(p) > 0])

print("Jumlah gambar ditemukan:", len(image_paths))
if len(image_paths) == 0:
    raise ValueError(f"Tidak ada gambar di folder: {INPUT_DIR}")

if TEST_MODE:
    image_paths = image_paths[:5]
    print("TEST MODE AKTIF → hanya 5 gambar")

# RESUME
done_set = set()
rows = []

if os.path.exists(OUTPUT_CSV_PATH):
    try:
        old = pd.read_csv(OUTPUT_CSV_PATH)
        if "filepath" in old.columns:
            done_set = set(old["filepath"].astype(str).tolist())
        rows = old.to_dict("records")
        print(f"Resume aktif: {len(done_set)} file sudah diproses.")
    except Exception as e:
        print("Gagal baca CSV lama:", e)

processed_since_save = 0

# MAIN LOOP
for img_path in tqdm(image_paths):

    if img_path in done_set:
        continue

    fname = os.path.basename(img_path)

    try:
        result = run_llm_on_image(img_path)

        # normalisasi label sedikit
        label = (result.get("Label") or "").strip().upper().replace(" ", "").replace("_", "-")
        if label == "NONSELFHARM":
            label = "NON-SELF-HARM"
        if label == "SELFHARM":
            label = "SELF-HARM"

        conf = (result.get("Confidance") or result.get("Confidence") or "").strip()
        if conf and not conf.endswith("%"):
            conf += "%"

        row = {
            "filename": fname,
            "Teks Terlihat": (result.get("Teks Terlihat") or "").strip(),
            "Label": label,
            "Confidance": conf,
            "Relasi Gambar dan Teks": (result.get("Relasi Gambar dan Teks") or "").strip(),
            "filepath": img_path
        }

    except Exception as e:
        print("ERROR DI FILE:", img_path)
        print("DETAIL:", type(e).__name__, e)

        row = {
            "filename": fname,
            "Teks Terlihat": "",
            "Label": "",
            "Confidance": "",
            "Relasi Gambar dan Teks": f"[ERROR] {type(e).__name__}: {e}",
            "filepath": img_path
        }

    rows.append(row)
    done_set.add(img_path)
    processed_since_save += 1

    if processed_since_save >= SAVE_EVERY:
        df = pd.DataFrame(rows)
        cols = ["filename", "Teks Terlihat", "Label", "Confidance", "Relasi Gambar dan Teks", "filepath"]
        df = df.reindex(columns=cols)
        safe_save_csv(df, OUTPUT_CSV_PATH)
        processed_since_save = 0

    time.sleep(DELAY_SECONDS)

# FINAL SAVE
df = pd.DataFrame(rows)
cols = ["filename", "Teks Terlihat", "Label", "Confidance", "Relasi Gambar dan Teks", "filepath"]
df = df.reindex(columns=cols)
safe_save_csv(df, OUTPUT_CSV_PATH)

print("SELESAI! CSV tersimpan di:", OUTPUT_CSV_PATH)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 45.0 MB/s eta 0:00:00
Mounted at /content/drive
Jumlah gambar ditemukan: 948


 38%|███▊      | 356/948 [24:44<41:08,  4.17s/it]


KeyboardInterrupt: 

In [ ]:
# INSTALL
!pip -q install --upgrade openai tqdm

import os, time, base64, glob, json
import pandas as pd
from tqdm import tqdm
from openai import OpenAI
from google.colab import drive

# MOUNT GOOGLE DRIVE
drive.mount('/content/drive')

# OPENAI API KEY
os.environ["OPENAI_API_KEY"] = "API-KEY"
client = OpenAI()

MODEL_NAME = "gpt-4o-mini"

# PATH
INPUT_DIR = "/content/drive/MyDrive/Thesis/dataset3"
OUTPUT_CSV_PATH = "/content/drive/MyDrive/Thesis/pseudo_label_gpt_ver3.csv"

# SETTINGS
DELAY_SECONDS = 1.5
SAVE_EVERY = 5
TEST_MODE = False # True = test 5 gambar, False = full run

# PROMPT
guideline_prompt = r"""
Anda adalah anotator ahli dalam psikologi, dan ahli linguistik. Tugas Anda adalah mengklasifikasikan meme menjadi [SELF-HARM] atau [NON-SELF-HARM] dengan presisi tinggi.
Ikuti alur penalaran (reasoning) di bawah ini secara berurutan. Jangan melompati langkah.
Langkah 1: Analisis Multimodal & Relasi
Lakukan analisis mendalam terhadap input dengan menjawab pertanyaan berikut:
1.  Analisis visual : apa objek, aktivitas, atau simbol yang muncul di gambar? apakah ada luka, darah, alat berbahaya, atau ekspresi emosional tertentu?
2.  Analisis tekstual: apa makna harfiah dari teks? apakah ada kata kunci terkait kematian, rasa sakit, atau keputusasaan?
3.  Analisis relasi (PENTING), yaitu bagaimana teks dan gambar berinteraksi? Apakah teks mengubah makna gambar netral menjadi berbahaya? Apakah ini sindiran (sarkasme), dark humor, atau hanya  ungkapan biasa ? Identifikasi resiko gabungan yang tidak terlihat jika teks/gambar dipisah.

Langkah 2: Indikator
Berdasarkan analisis relasi di atas, periksa apakah meme ini memenuhi salah satu indikator valid berikut secara ketat:
A. Indikator Narasi & Teks
 (1a) Metode dan deskripsi, mendeskripsikan cara menyakiti diri (cutting, burning, overdosis)
 (1b) Masking & romantisasi, menganggap self-harm estetik, puitis, atau mekanisme coping yang wajar
(1c)  Ideasi bunuh diri, menyiratkan keinginan mati
(1d) Gaslighting/manipulasi, menyalahkan orang lain atas niat menyakiti diri sendiri.
B. Indikator visual dan simbolik
 (2a) Luka fisik,Gambar darah, bekas sayatan, lebam.
(2b) Alat berbahaya seperti Silet, pisau, tali gantung, tumpukan obat
 (2c) Estetika dark humor, gambar ceria/kartun imut tetapi teksnya tentang keinginan mati/bunuh diri (kontras ironis)
Instruksi Label:
 Jika “terdapat minimal satu” indikator terpenuhi (terutama indikator relasional 2c) -> Klasifikasikan sebagai [SELF-HARM].
 Jika tidak ada indikator terpenuhi dan konteks hanya humor umum -> Klasifikasikan sebagai [NON-SELF-HARM].

Langkah 3: Self-checking
Sebelum memberikan label final, kritik keputusan Anda sendiri:
 "Apakah saya salah mengartikan *slang* atau lelucon internet sebagai ancaman serius?"
 "Apakah saya mengabaikan sinyal bahaya terselubung karena gambarnya terlihat lucu (kartun)?"
 Jika ragu, prioritaskan keselamatan (safety-first approach) namun tetap bedakan antara dark jokes  umum vs promosi self-harm spesifik.
"""

# Tambahan instruksi
json_instruction = r"""
Keluarkan output HANYA dalam format JSON object valid dengan key persis berikut:
{
  "Teks Terlihat": "<teks di gambar apa adanya>",
  "Label": "SELF-HARM" atau "NON-SELF-HARM",
  "Confidance": "<angka 0-100% (pakai tanda %)>",
  "Relasi Gambar dan Teks": "<1 kalimat singkat relasi teks-gambar>"
}
Jangan tambahkan teks lain di luar JSON.
"""

def encode_image_to_data_url(image_path: str) -> str:
    with open(image_path, "rb") as f:
        encoded = base64.b64encode(f.read()).decode("utf-8")

    ext = os.path.splitext(image_path.lower())[1]
    if ext == ".png":
        mime = "image/png"
    elif ext in [".jpg", ".jpeg"]:
        mime = "image/jpeg"
    elif ext == ".webp":
        mime = "image/webp"
    else:
        mime = "image/png"

    return f"data:{mime};base64,{encoded}"

def run_llm_on_image(image_path: str) -> dict:
    image_data_url = encode_image_to_data_url(image_path)

    resp = client.chat.completions.create(
        model=MODEL_NAME,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": "Kamu adalah annotator multimodal yang teliti dan mengikuti format output."},
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": guideline_prompt.strip() + "\n\n" + json_instruction.strip()},
                    {"type": "image_url", "image_url": {"url": image_data_url}},
                ],
            },
        ],
        temperature=0,
        max_tokens=500
    )

    content = resp.choices[0].message.content
    return json.loads(content)

def safe_save_csv(df: pd.DataFrame, out_path: str):
    tmp_path = out_path + ".tmp"
    df.to_csv(tmp_path, index=False)
    os.replace(tmp_path, out_path)

# COLLECT IMAGE FILES
image_exts = ("*.png", "*.jpg", "*.jpeg", "*.webp")
image_paths = []
for pat in image_exts:
    image_paths.extend(glob.glob(os.path.join(INPUT_DIR, pat)))

image_paths = sorted([p for p in image_paths if os.path.getsize(p) > 0])

print("Jumlah gambar ditemukan:", len(image_paths))
if len(image_paths) == 0:
    raise ValueError(f"Tidak ada gambar di folder: {INPUT_DIR}")

if TEST_MODE:
    image_paths = image_paths[:5]
    print("TEST MODE AKTIF → hanya 5 gambar")

# RESUME
done_set = set()
rows = []

if os.path.exists(OUTPUT_CSV_PATH):
    try:
        old = pd.read_csv(OUTPUT_CSV_PATH)
        if "filepath" in old.columns:
            done_set = set(old["filepath"].astype(str).tolist())
        rows = old.to_dict("records")
        print(f"Resume aktif: {len(done_set)} file sudah diproses.")
    except Exception as e:
        print("Gagal baca CSV lama:", e)

processed_since_save = 0

# MAIN LOOP
for img_path in tqdm(image_paths):

    if img_path in done_set:
        continue

    fname = os.path.basename(img_path)

    try:
        result = run_llm_on_image(img_path)

        # normalisasi label sedikit
        label = (result.get("Label") or "").strip().upper().replace(" ", "").replace("_", "-")
        if label == "NONSELFHARM":
            label = "NON-SELF-HARM"
        if label == "SELFHARM":
            label = "SELF-HARM"

        conf = (result.get("Confidance") or result.get("Confidence") or "").strip()
        if conf and not conf.endswith("%"):
            conf += "%"

        row = {
            "filename": fname,
            "Teks Terlihat": (result.get("Teks Terlihat") or "").strip(),
            "Label": label,
            "Confidance": conf,
            "Relasi Gambar dan Teks": (result.get("Relasi Gambar dan Teks") or "").strip(),
            "filepath": img_path
        }

    except Exception as e:
        print("ERROR DI FILE:", img_path)
        print("DETAIL:", type(e).__name__, e)

        row = {
            "filename": fname,
            "Teks Terlihat": "",
            "Label": "",
            "Confidance": "",
            "Relasi Gambar dan Teks": f"[ERROR] {type(e).__name__}: {e}",
            "filepath": img_path
        }

    rows.append(row)
    done_set.add(img_path)
    processed_since_save += 1

    if processed_since_save >= SAVE_EVERY:
        df = pd.DataFrame(rows)
        cols = ["filename", "Teks Terlihat", "Label", "Confidance", "Relasi Gambar dan Teks", "filepath"]
        df = df.reindex(columns=cols)
        safe_save_csv(df, OUTPUT_CSV_PATH)
        processed_since_save = 0

    time.sleep(DELAY_SECONDS)

# FINAL SAVE
df = pd.DataFrame(rows)
cols = ["filename", "Teks Terlihat", "Label", "Confidance", "Relasi Gambar dan Teks", "filepath"]
df = df.reindex(columns=cols)
safe_save_csv(df, OUTPUT_CSV_PATH)

print("SELESAI! CSV tersimpan di:", OUTPUT_CSV_PATH)


Mounted at /content/drive
Jumlah gambar ditemukan: 593


100%|██████████| 593/593 [39:29<00:00,  4.00s/it]

SELESAI! CSV tersimpan di: /content/drive/MyDrive/Thesis/pseudo_label_gpt_ver3.csv
